In [ ]:
import time
import numpy as np

print("=" * 66)
print("DEMO: PYTHON LOOPS vs NUMPY BLAS")

N_small = 160
A_list = [[1.0] * N_small for _ in range(N_small)]
B_list = [[2.0] * N_small for _ in range(N_small)]
C_list = [[0.0] * N_small for _ in range(N_small)]
t0 = time.perf_counter()
for i in range(N_small):
    for j in range(N_small):
        s = 0.0
        for k in range(N_small):
            s += A_list[i][k] * B_list[k][j]
        C_list[i][j] = s
t_python = time.perf_counter() - t0
print(f"  Pure Python {N_small}x{N_small}: {t_python:.4f}s")

# same-size comparison (fair), then a big one for GFLOP/s
A_s = np.ones((N_small, N_small)); B_s = np.full((N_small, N_small), 2.0)
_ = A_s @ B_s                                   # warm-up
t0 = time.perf_counter(); C_s = A_s @ B_s; t_np_small = time.perf_counter() - t0
print(f"  NumPy       {N_small}x{N_small}: {t_np_small:.6f}s   -> {t_python / max(t_np_small, 1e-9):,.0f}x faster than the loops")
assert abs(C_s[0, 0] - C_list[0][0]) < 1e-9

N_large = 1024
A_np = np.ones((N_large, N_large), dtype=np.float64)
B_np = np.full((N_large, N_large), 2.0, dtype=np.float64)
_ = np.dot(A_np[:8, :8], B_np[:8, :8])
t0 = time.perf_counter()
C_np = np.dot(A_np, B_np)
t_mkl = time.perf_counter() - t0
gflops = (2.0 * (N_large ** 3) / max(t_mkl, 1e-9)) / 1e9
print(f"  NumPy BLAS  {N_large}x{N_large}: {t_mkl:.4f}s  ({gflops:.2f} GFLOP/s)")
print(f"  Check: C[0,0] = {C_np[0,0]:.1f}  (expect {float(N_large * 2):.1f})")
assert abs(C_np[0, 0] - (N_large * 2.0)) < 1e-6
print("=" * 66)

DEMO: PYTHON LOOPS vs NUMPY BLAS
  Pure Python 160x160: 1.3219s
  NumPy       160x160: 0.003863s   -> 342x faster than the loops
  NumPy BLAS  1024x1024: 0.0805s  (26.68 GFLOP/s)
  Check: C[0,0] = 2048.0  (expect 2048.0)
